# Aula 1 - LLM APIs

## Template da V1 do projeto

Neste notebook, você vai montar a primeira versão do assistente de estudos da monitoria.

As células com `TODO` são partes que você precisa completar. Não coloque sua API key diretamente no notebook, utilize o `.env`.

## Objetivos

Ao final, você deve conseguir:

- configurar um cliente para a API da Groq;
- montar mensagens com `system` e `user`;
- enviar uma requisição e ler a resposta;
- observar o uso de tokens;
- explicar por que chamadas separadas não têm memória automática;
- testar o efeito da temperatura;
- salvar a V1 em um commit.

## 1. Configuração do ambiente

Antes de executar o notebook:

1. crie um ambiente virtual;
2. instale as dependências de `requirements.txt`;
3. copie `env.example` para `.env`;
4. preencha `GROQ_API_KEY` no `.env`.

In [1]:
messages = [
    {"role": "system", "content": "Tu és um assistente para dúvidas de monitoria."},
    {"role": "user", "content": "Como posso tirar dúvidas na monitoria?"}
]

In [2]:
# Passo a passo para criar o .venv: 
# Abra o Terminal na raiz do projeto e digite:
# python -m venv venv
# Depois digite: 
# cd "Aula 01"
# pip install -r requirements.txt

import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")
if not api_key:
    raise RuntimeError(
        "GROQ_API_KEY não encontrada. Copie env.example para .env e informe sua chave."
    )

client = OpenAI(
    api_key=api_key,
    base_url=os.getenv("GROQ_BASE_URL", "https://api.groq.com/openai/v1"),
)

MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-120b")
print(f"Modelo configurado: {MODEL}")

Modelo configurado: openai/gpt-oss-120b


## 2. Monte as mensagens

A aplicação controla o comportamento do assistente usando mensagens.

Use uma mensagem `system` para definir o papel do assistente e uma mensagem `user` para fazer uma pergunta sobre a monitoria.

In [3]:
messages = [
    {
        "role" : "system", 
        "content": "Voce é um assistente para dúvidas de monitoria."
    },
    {
        "role": "user",
        "content": "Como posso tirar dúvidas na monitoria?"
    }
]

## 3. Faça a primeira chamada

Envie `messages` para o modelo configurado em `MODEL` e salve a resposta em uma variável chamada `response`.

In [4]:
response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=0,
)

## 4. Leia o texto da resposta

A resposta contém uma lista de escolhas. Nesta atividade, vamos usar o texto da primeira escolha.

In [5]:

print(response.choices[0].message.content)

Claro! Tirar dúvidas na monitoria pode ser bem simples se você seguir alguns passos que facilitam a comunicação e garantem que você receba a ajuda de que precisa. Aqui vão algumas dicas práticas:

---

## 1. **Escolha o canal adequado**
| Canal | Quando usar | Como acessar |
|-------|-------------|--------------|
| **Sala de aula virtual (Zoom, Teams, Google Meet)** | Quando a dúvida é urgente ou requer explicação ao vivo. | Verifique o link enviado pelo monitor ou pelo professor. |
| **Fórum ou grupo de mensagens (Discord, Slack, WhatsApp, Telegram)** | Para dúvidas que podem ser respondidas de forma assíncrona. | Entre no grupo indicado no plano de monitoria. |
| **E‑mail ou formulário online** | Quando a questão exige envio de arquivos (código, PDF, planilha). | Use o endereço ou link fornecido no material da disciplina. |
| **Horário de plantão presencial** | Se a monitoria tem encontros presenciais e você prefere explicação cara a cara. | Consulte o calendário da disciplina. |

--

## 5. Observe o uso de tokens

Além do texto, a API informa quantos tokens entraram na chamada, quantos saíram e qual foi o total.

In [6]:
print("Tokens do prompt (entrada):", response.usage.prompt_tokens)
print("Tokens da resposta (saída):", response.usage.completion_tokens)
print("Total de tokens usados:", response.usage.total_tokens)

Tokens do prompt (entrada): 94
Tokens da resposta (saída): 1203
Total de tokens usados: 1297


## 6. Teste a ausência de memória

Faça uma chamada dizendo que o nome do seu usuário. Depois faça uma segunda chamada perguntando o nome, mas sem reenviar a primeira mensagem.

Salve a segunda resposta em `sem_historico`. O que aconteceu?

In [7]:

client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Olá! O meu nome é João."}
    ]
)


sem_historico = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Qual é o meu nome?"}
    ]
)


print(sem_historico.choices[0].message.content)

Não sei qual é o seu nome, já que você ainda não o compartilhou comigo. Se quiser, pode me dizer como prefere ser chamado, ou me contar como posso ajudar você hoje!


As APIs de modelos de linguagem (LLMs) não possuem memória interna entre chamadas independentes. Na chamada sem histórico, o modelo não sabe o meu nome porque cada requisição é tratada do zero. Para simular a sensação de memória num chatbot, a aplicação é responsável por re-enviar todo o histórico da conversa (incluindo as mensagens de papel assistant) em cada nova requisição.

In [8]:
# print(sem_historico.choices[0].message.content)

Agora repita a pergunta, mas envie explicitamente o histórico em `messages`. Inclua uma mensagem `assistant` representando a resposta anterior.

A aplicação cria a sensação de memória ao reenviar o contexto.

In [9]:

mensagens_com_historico = [
    {"role": "user", "content": "Olá! O meu nome é João."},
    {"role": "assistant", "content": "Olá João! Em que posso ajudar?"},
    {"role": "user", "content": "Qual é o meu nome?"}
]


com_historico = client.chat.completions.create(
    model=MODEL,
    messages=mensagens_com_historico
)

In [10]:
# Imprimir a resposta obtida com o histórico incluído
print(com_historico.choices[0].message.content)

O seu nome é João. Como posso ajudar?


## 7. Experimente a temperatura

Use o mesmo prompt duas vezes: uma com `temperature=0` e outra com `temperature=1`.

Compare a variedade das respostas. Temperatura baixa tende a ser mais previsível; temperatura alta tende a permitir mais variação.

In [11]:

prompt_teste = [
    {"role": "user", "content": "Dá-me 3 ideias criativas de nomes para uma aplicação de inteligência artificial."}
]


resposta_temp_0 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_teste,
    temperature=0
)


resposta_temp_1 = client.chat.completions.create(
    model=MODEL,
    messages=prompt_teste,
    temperature=1
)


print("temperatura 0 (Previsível)")
print(resposta_temp_0.choices[0].message.content)

print("\n" + "="*35 + "\n")

print("temperatura 1 (Variada/Criativa)")
print(resposta_temp_1.choices[0].message.content)

temperatura 0 (Previsível)
Claro! Aqui vão três sugestões criativas de nomes para a sua aplicação de inteligência artificial:

1. **NeuroMente** – combina a ideia de redes neurais com a capacidade de “pensar” e entender o usuário, transmitindo inteligência e empatia.

2. **SynaptiQ** – um trocadilho entre “sinapse” (conexões cerebrais) e “IQ” (quociente de inteligência), sugerindo uma IA que aprende e se adapta rapidamente.

3. **AetherAI** – “Éter” remete a algo etéreo, onipresente e avançado; o sufixo “AI” deixa claro o foco em inteligência artificial, criando um nome elegante e futurista.


temperatura 1 (Variada/Criativa)
Claro! Aqui vão três sugestões de nomes criativos para a sua aplicação de inteligência artificial, acompanhados de uma breve explicação de cada um:

| # | Nome | Por que funciona |
|---|------|-------------------|
| 1 | **NeuroPulse** | Evoca a ideia de “pulso” ou “batimento” de um cérebro artificial, sugerindo rapidez, dinamismo e inteligência em tempo real. |
| 

A temperatura controla a aleatoriedade da geração do texto:
- Com temperature=0: O modelo gera respostas focadas, previsíveis e determinísticas (sempre muito semelhantes para o mesmo prompt).
- Com temperature=1: O modelo ganha variabilidade e criatividade, podendo trazer ideias e estruturas de frases diferentes a cada execução.

## 8. Checkpoint da V1

Antes de fazer o commit, confirme:

- [ x] a aplicação chamou a API da Groq;
- [x ] a chave ficou no `.env`;
- [x ] a requisição usou `system` e `user`;
- [x ] a resposta apareceu;
- [ x] o uso de tokens foi observado;
- [ x] você comparou chamadas sem histórico e com histórico;
- [ x] você testou duas temperaturas.
